# Dataset : Prédiction du Maïs en Afrique

Dataset pour **tous les pays d'Afrique** (34 pays).

## 1. Importations et Liste des Pays

In [21]:
import pandas as pd

## 2. CHARGER LES DONNÉES RÉELLES FAOSTAT

In [22]:
dataset = pd.read_csv("./data/africa_maize.csv")
print(f"Shape : {dataset.shape}")
dataset.head()

Shape : (988, 5)


,country,year,production_tonnes,yield_t_ha,producer_price
0,Algeria,1991,500.0,1923.1,173.2
1,Algeria,1992,662.0,2282.8,522.1
2,Algeria,1993,225.0,1250.0,488.3
3,Algeria,1994,185.0,451.2,325.2
4,Algeria,1995,419.0,1611.5,335.7


Colonne du datatset 

In [23]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 988 entries, 0 to 987
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   country            988 non-null    object 
 1   year               988 non-null    int64  
 2   production_tonnes  988 non-null    float64
 3   yield_t_ha         988 non-null    float64
 4   producer_price     988 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 38.7+ KB


Valeurs Manquantes

In [24]:
dataset.isnull().sum()

country              0
year                 0
production_tonnes    0
yield_t_ha           0
producer_price       0
dtype: int64

In [25]:
years = range(dataset["year"].min(), dataset["year"].max() + 1)
countries = dataset["country"].unique()

In [26]:
full_index = pd.MultiIndex.from_product(
    [countries, years],
    names=["country", "year"]
)

df_full = pd.DataFrame(index=full_index).reset_index()

In [27]:
dataset = df_full.merge(dataset, on=["country", "year"], how="left")
dataset.head(10)

,country,year,production_tonnes,yield_t_ha,producer_price
0,Algeria,1991,500.0,1923.1,173.2
1,Algeria,1992,662.0,2282.8,522.1
2,Algeria,1993,225.0,1250.0,488.3
3,Algeria,1994,185.0,451.2,325.2
4,Algeria,1995,419.0,1611.5,335.7
5,Algeria,1996,NaN,NaN,NaN
6,Algeria,1997,NaN,NaN,NaN
7,Algeria,1998,NaN,NaN,NaN
8,Algeria,1999,NaN,NaN,NaN
9,Algeria,2000,NaN,NaN,NaN


In [28]:
cols = ["production_tonnes", "yield_t_ha", "producer_price"]

valid_countries = dataset.groupby("country")[cols].apply(
    lambda x: x.notna().sum().sum() > 0
)

valid_countries = valid_countries[valid_countries].index

dataset = dataset[dataset["country"].isin(valid_countries)]

dataset = dataset.sort_values(["country", "year"])

dataset[cols] = dataset.groupby("country")[cols].transform(
    lambda x: x.interpolate(method="linear")
)


In [29]:
dataset[cols] = dataset.groupby("country")[cols].transform(
    lambda x: x.ffill().bfill()
)

In [30]:
dataset.to_csv("./data/maize_final.csv", index=False)